# Phase 15 Promoted Assertion Epistemics

Journey purpose: prove the broadened promoted-assertion epistemic slice over the recovered graph layer.

Notebook mode: proof

Related docs: `docs/plans/0001_successor_roadmap.md`, `docs/plans/0010_phase15_epistemic_corroboration_shape.md`, `docs/adr/0016-recover-phase-15-through-extension-local-promoted-assertion-dispositions-and-derived-corroboration.md`

Related tests: `tests/extensions/test_epistemic_service.py`, `tests/integration/test_epistemic_cli.py`

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from tempfile import TemporaryDirectory

from onto_canon6.core import CanonicalGraphService
from onto_canon6.extensions.epistemic import EpistemicService
from onto_canon6.pipeline import ReviewService
from onto_canon6.surfaces import EpistemicReportService

workspace = TemporaryDirectory()
review_db_path = Path(workspace.name) / 'review.sqlite3'
review_service = ReviewService(
    db_path=review_db_path,
    overlay_root=Path(workspace.name) / 'ontology_overlays',
    default_acceptance_policy='record_only',
)
graph_service = CanonicalGraphService(db_path=review_db_path)
epistemic_service = EpistemicService(db_path=review_db_path)

def submit_and_accept(*, payload: dict[str, object], source_ref: str) -> str:
    submission = review_service.submit_candidate_assertion(
        payload=payload,
        profile_id='default',
        profile_version='1.0.0',
        submitted_by='analyst:notebook',
        source_kind='note',
        source_ref=source_ref,
    )
    return review_service.review_candidate(
        candidate_id=submission.candidate.candidate_id,
        decision='accepted',
        actor_id='analyst:reviewer',
    ).candidate_id


## Phase 1: Promoted Assertion Setup

Purpose: create one corroboration pair and one tension pair over promoted assertions.

`input -> output`: accepted candidates -> promoted assertions

Acceptance criteria:
- promoted assertions exist for both corroboration and tension examples
- the setup is fully live against the persisted review and graph stores

Status: `proven`

Execution mode: `live`

In [2]:
corroborating_payload = {
    'predicate': 'oc:hold_command_role',
    'roles': {
        'commander': [{'entity_id': 'ent:person:eric_olson'}],
        'organization': [{'entity_id': 'ent:org:ussocom'}],
        'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}],
    },
}

corroborating_a = submit_and_accept(payload=corroborating_payload, source_ref='notes/corroborating-a.txt')
corroborating_b = submit_and_accept(payload=corroborating_payload, source_ref='notes/corroborating-b.txt')
tension_a = submit_and_accept(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:john_smith'}],
            'organization': [{'entity_id': 'ent:org:joint_staff'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}],
        },
    },
    source_ref='notes/tension-a.txt',
)
tension_b = submit_and_accept(
    payload={
        'predicate': 'oc:hold_command_role',
        'roles': {
            'commander': [{'entity_id': 'ent:person:john_smith'}],
            'organization': [{'entity_id': 'ent:org:joint_staff'}],
            'title': [{'kind': 'value', 'value_kind': 'string', 'value': 'Director'}],
        },
    },
    source_ref='notes/tension-b.txt',
)

promoted = [
    graph_service.promote_candidate(candidate_id=candidate_id, promoted_by='analyst:graph-promoter')
    for candidate_id in (corroborating_a, corroborating_b, tension_a, tension_b)
]
[result.assertion.assertion_id for result in promoted]


['gassert_560047cfef6093021f7fdf50',
 'gassert_6ab34acb0fa7ef3657043e78',
 'gassert_a426345db2951b7382d3754d',
 'gassert_11ffb3c50bc809c486f4fe67']

## Phase 2: Assertion Disposition

Purpose: record explicit promoted-assertion disposition history without mutating graph tables.

`input -> output`: promoted assertion -> disposition event

Acceptance criteria:
- a promoted assertion can be weakened explicitly
- the disposition is persisted through the extension-local store

Status: `proven`

Execution mode: `live`

In [3]:
tension_assertion_id = promoted[2].assertion.assertion_id
disposition = epistemic_service.record_assertion_disposition(
    assertion_id=tension_assertion_id,
    target_status='weakened',
    actor_id='analyst:epistemic',
    rationale='Conflicting title evidence lowers certainty but does not retract the claim.',
)
disposition


AssertionDispositionRecord(disposition_id='edisp_9bcbfca061e446d6a1477542', assertion_id='gassert_a426345db2951b7382d3754d', prior_status='active', target_status='weakened', actor_id='analyst:epistemic', rationale='Conflicting title evidence lowers certainty but does not retract the claim.', created_at='2026-03-18T21:17:05.255982+00:00')

## Phase 3: Corroboration And Tension Report

Purpose: inspect the broadened promoted-assertion epistemic report.

`input -> output`: promoted assertions + epistemic events -> typed collection report

Acceptance criteria:
- the report shows one corroboration group
- the report shows one tension pair
- the weakened assertion carries explicit status in the report

Status: `proven`

Execution mode: `live`

In [4]:
report = EpistemicReportService(epistemic_service=epistemic_service).build_promoted_assertion_collection_report()
assert report.summary.total_assertions == 4
assert report.summary.total_weakened == 1
assert report.summary.total_corroboration_groups == 1
assert report.summary.total_tension_pairs == 1
report


PromotedAssertionEpistemicCollectionReport(assertion_reports=(PromotedAssertionEpistemicReport(assertion=PromotedGraphAssertionRecord(assertion_id='gassert_560047cfef6093021f7fdf50', source_candidate_id='cand_1fca1f440d4d4dbe85df423c', profile=ProfileRef(profile_id='default', profile_version='1.0.0'), predicate='oc:hold_command_role', normalized_body={'predicate': 'oc:hold_command_role', 'roles': {'commander': [{'entity_id': 'ent:person:eric_olson', 'kind': 'entity'}], 'organization': [{'entity_id': 'ent:org:ussocom', 'kind': 'entity'}], 'title': [{'kind': 'value', 'value': 'Commander', 'value_kind': 'string'}]}}, claim_text=None, promoted_by='analyst:graph-promoter', promoted_at='2026-03-18T21:17:05.225335+00:00'), epistemic_status='active', confidence=None, current_disposition=None, disposition_history=(), superseded_by=None, superseded_by_assertion_id=None, corroborating_assertion_ids=('gassert_6ab34acb0fa7ef3657043e78',), tensions=()), PromotedAssertionEpistemicReport(assertion=Pro

## Current Gap Or Promotion Path

Temporal and inference behavior are explicitly deferred from the current successor scope.
This notebook proves the narrower promoted-assertion epistemic slice without reintroducing a general truth-maintenance runtime.

In [5]:
workspace.cleanup()